# Search Engine 

# with tools and agents

In [93]:
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [1]:
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.agents import create_tool_calling_agent,AgentExecutor
from langchain_community.tools import WikipediaQueryRun,ArxivQueryRun,DuckDuckGoSearchRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import AgentType
from langchain_classic.document_loaders import PyPDFLoader




d:\Gen AI\LangChain\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [94]:
pip install wikipedia 


Note: you may need to restart the kernel to use updated packages.


In [95]:
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=2000)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [96]:
pip install duckduckgo-search

Note: you may need to restart the kernel to use updated packages.


In [97]:
%pip install -U ddgs


Note: you may need to restart the kernel to use updated packages.


In [98]:
from langchain_community.tools import DuckDuckGoSearchRun

ddg_tool = DuckDuckGoSearchRun()

print(ddg_tool.run("Latest Llama model releases"))

4 days ago · Stay updated with open source LLM updates today covering models like Llama 3, Mistral, Qwen, and DeepSeek —now rivaling proprietary alternatives on many benchmarks while providing flexibility to fine-tune, self-host, and customize for specific domains. 17 hours ago · Understanding the Llama Model Family Open Source & Free Meta releases Llama model weights under permissive open licenses, making them free to download, modify, and deploy commercially. This open approach has made Llama the most widely adopted open-source LLM family, powering thousands of applications, research projects, and fine-tuned variants. Sep 2, 2026 · On August 10, 2026 it released Muse Glimmer 30B, a local agent model with open weights under Apache 2.0 — Meta's first downloadable frontier-lab model since Llama 4, and the first ever to ship without the Llama license's >700M-MAU and EU carve-outs. LLM inference in C/C++. Contribute to ggml-org/llama.cpp development by creating an account on GitHub. 17 ho

In [99]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv,handle_tool_error=True)
arxiv.name


'arxiv'

In [100]:
tools=[wiki,arxiv,ddg_tool]

In [101]:
##Custom tools[RAG tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [102]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
embeddings=OllamaEmbeddings(model="nomic-embed-text")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=10000,chunk_overlap=300)
text_splits=documents.split_documents(docs)
vectordb=FAISS.from_documents(text_splits,embeddings)
retriever=vectordb.as_retriever()
retriever


VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000026EC4D9D930>, search_kwargs={})

In [103]:
from langchain_core.tools import  create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith-search","search anything about langsmith")
retriever_tool.name

'langsmith-search'

In [104]:
tools=[wiki,arxiv,ddg_tool,retriever_tool]

In [105]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\Gen AI\\LangChain\\env\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=2000)),
 ArxivQueryRun(handle_tool_error=True, api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.arxiv.ArxivError'>, <class 'arxiv.arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text')),
 StructuredTool(name='langsmith-search', description='search anything about langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<l

# # run all this tools with agents and llm models

In [106]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key=os.getenv("GROQ_API_KEY")
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
llm = ChatGroq(groq_api_key=api_key, model_name="openai/gpt-oss-20b")

In [107]:
pip install langchainhub

Note: you may need to restart the kernel to use updated packages.


In [108]:
## prompt template
from langchainhub import Client

client = Client()
prompt = client.pull("hwchase17/openai-functions-agent")
prompt

C:\Users\viswa\AppData\Local\Temp\ipykernel_4912\2907425436.py:5: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = client.pull("hwchase17/openai-functions-agent")
d:\Gen AI\LangChain\env\lib\site-packages\langchainhub\client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)


'{"id": ["langchain", "prompts", "chat", "ChatPromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"messages": [{"id": ["langchain", "prompts", "chat", "SystemMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "You are a helpful assistant", "input_variables": [], "template_format": "f-string", "partial_variables": {}}}}}, {"id": ["langchain", "prompts", "chat", "MessagesPlaceholder"], "lc": 1, "type": "constructor", "kwargs": {"optional": true, "variable_name": "chat_history"}}, {"id": ["langchain", "prompts", "chat", "HumanMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "{input}", "input_variables": ["input"], "template_format": "f-string", "partial_variables": {}}}}}, {"id": ["langchain", "pr

In [109]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import create_openai_tools_agent

# Define the template properly as a ChatPromptTemplate object
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Verify it is no longer a string
print(type(prompt))
agent = create_openai_tools_agent(llm,tools,prompt)

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>


In [110]:
# Agent Executor
from langchain_classic.agents import AgentExecutor
agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [111]:
agent_executor.invoke({"input":"tell me about langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'LangSmith'}`


LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failures, monitor quality, and build the datasets you evaluate against.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic A

{'input': 'tell me about langsmith',
 'output': '**LangSmith** is a platform designed to give you full visibility into the behavior, performance, and quality of your large‑language‑model (LLM) applications. Think of it as a “telemetry & observability” layer you can drop into any LLM‑powered workflow—whether you’re building chatbots, RAG pipelines, agents, or custom LLM services.\n\n---\n\n### Core Capabilities\n\n| Feature | What it does | Why it matters |\n|---------|--------------|----------------|\n| **Tracing** | Records every call, prompt, response, and internal step in your LLM workflow. | Lets you see exactly what happened in a given run, debug failures, and audit usage. |\n| **Debug & Inspect** | UI for viewing threads, runs, messages, and turns. | Quickly spot where a model diverged from expected behavior. |\n| **Metrics & Dashboards** | Aggregated performance metrics (latency, cost, token counts, success rates). | Monitor health in real time, set alerts for anomalies. |\n| **

In [112]:
agent_executor.invoke({"input":"whats the papaer arXiv:1706.03762 about?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': 'arXiv:1706.03762'}`




HTTPError: Page request resulted in HTTP 301: Gated Linear Units (arXiv:1612.08083) consist of the component-wise product of two linear projections, one of which is first passed through a sigmoid function. Variations on GLU are possible, using different nonlinear (or even linear) functions in place of sigmoid. We test these variants in the feed-forward sublayers of the Transformer (arXiv:1706.03762) sequence-to-sequence model, and find that some of them yield quality improvements over the typically-used ReLU or GELU activations. (http://export.arxiv.org/api/query?search_query=arXiv%3A1706.03762&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=1)

In [113]:
import streamlit as st
from langchain_groq import ChatGroq
from langchain_classic.community import WikipediaAPIWrapper,ArxivAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun,WikipediaQueryRun,ArxivQueryRun
from langchain.agents import initialize_agent,AgentType
from langchain.callbacks import StreamlitCallbackHandler


ModuleNotFoundError: No module named 'streamlit'